# Offline Pokémon PPO lab — Macintosh / M5 Pro

This lab tests the instability hypotheses in the [Reddit post and comments](https://www.reddit.com/r/reinforcementlearning/comments/197zbyt/ppo_agent_playing_pokemon_showdown_vs_random/): learning rate, entropy, collection size, and completed episode returns. Read the [research notes](../docs/offline-ppo-experiments-research.md) for sources and limits.

Run cells from top to bottom with **Pokemon Offline (M5 Pro)**. Existing results load without starting another experiment. Set the run switches below to `True` when ready. Every new run gets its own directory under `artifacts/`; copied data lives under `data/`. No login or live checkpoint promotion happens here.

The main comments were read. Six expandable replies could not be retrieved; direct Reddit JSON returned HTTP 403. This is an adaptation to the project's Champions VGC simulator, terminal rewards and complete games; the original post did not provide its reward, format or code.

In [ ]:
from pathlib import Path
import os, sys, json, gzip, sqlite3, time
from dataclasses import asdict, replace
from datetime import datetime
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "ml/model.py").is_file())
assert (ROOT / "ml/model.py").exists(), "Start Jupyter from the project directory"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["PS_TORCH_THREADS"] = "4"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "data/jupyter/matplotlib"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.offline_ppo import (
    ARMS, CORPUS, FMT, Compute, inventory, prepare_teams,
    run_matrix, benchmark_workers, benchmark_training,
    prepare_scout, compare_scout, tables, write_json,
)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": .2})
display(pd.Series(inventory(), name="Macintosh"))
assert Path(sys.executable).resolve() == (ROOT / ".venv/bin/python").resolve(), "Select the project kernel"
ARCHIVED = ROOT / "data/research-archive/macintosh-offline-ppo-20261009"
DATA_ROOT = ARCHIVED if ARCHIVED.exists() else ROOT
CORPUS = DATA_ROOT / "data/offline-corpus/divemac-20261009"

## What the project learns

The local Node Showdown simulator emits each player's own request and public information. A PyTorch actor scores complete legal joint actions; a critic predicts the outcome. The actor starts with a mechanical heuristic prior. The initial checkpoint's score is therefore the learning baseline.

PPO uses accepted sampled decisions from a **frozen collecting revision**, checks their original likelihoods, computes GAE, and uses each completed batch once. Terminal rewards are win `+1`, loss `−1`, draw `0`; unfinished games get no outcome. For these rewards, average episode return equals `(wins − losses) / completed games`.

Real public replays teach a separate opponent scout to predict **executed public moves from pre-turn information**. They do not supply the private requests or current policy probabilities PPO requires. Historical M-B and current M-C remain separate.

In [ ]:
manifest = json.loads((CORPUS / "manifest.json").read_text())
display(pd.Series(manifest["selected_counts"], name="copied records"))
display(pd.DataFrame(manifest["inventory"]["episodes"]))
display(pd.DataFrame(manifest["inventory"]["public_battles"]))
print(f"Verified export: {sum(f['bytes'] for f in manifest['files'].values()) / 1e6:.1f} MB")
print("Source commit:", manifest["source_commit"])
display(manifest["licenses"])
# Read-only queries of the copied database; never open the remote live WAL files.
database_path = CORPUS / "inputs/experience.sqlite3"
if not database_path.exists() and database_path.with_suffix(".sqlite3.gz").exists():
    import tempfile, shutil
    temporary_corpus = tempfile.TemporaryDirectory(prefix="pokemon-notebook-corpus-")
    database_path = Path(temporary_corpus.name) / "experience.sqlite3"
    with gzip.open(CORPUS / "inputs/experience.sqlite3.gz", "rb") as src, database_path.open("wb") as dst:
        shutil.copyfileobj(src, dst)
corpus_db = sqlite3.connect(database_path.as_uri() + "?mode=ro", uri=True)
display(pd.read_sql_query("SELECT format, count(*) AS battles FROM public_battles GROUP BY format", corpus_db))
corpus_db.close()
if "temporary_corpus" in globals():
    temporary_corpus.cleanup()

## Experiment and resource controls

`full` keeps 480/1,024 fresh completed-decision targets from the research matrix, runs 30 updates for each of three independent seeds, and evaluates 100 games per opponent. `pilot` uses 128 decisions, three updates, one seed, and eight evaluation games; it validates execution and is too small for strength claims. `smoke` uses one update and four evaluation games.

The **intensive** profile uses full training duty and can request up to 14 simulator workers on this 15-core Mac. Each worker uses one PyTorch/BLAS thread. Available memory limits actual wave concurrency; at least 2 GiB remains available and Metal allocation stays bounded. These settings are isolated from DiveMac's background service budgets. Stop a long run with Kernel → Interrupt; artifacts remain for diagnosis. Interrupted matrices require a fresh output directory.

Benchmark worker counts and actual PPO training on CPU/MPS before selecting the sustained configuration. The CPU runs battle mechanics; Metal runs optimizer work. GPU availability does not establish higher throughput.

In [ ]:
MODE = "pilot"                    # "smoke", "pilot", or "full"
RUN_MATRIX = False                 # True starts fresh offline experiments
RUN_WORKER_BENCHMARK = False       # CPU simulator concurrency sweep
RUN_TRAINING_BENCHMARK = False     # Real completed-rollout CPU vs Metal PPO
RUN_SCOUT = False                  # Re-extract labels and fit copied real battles
RUN_SCOUT_COMPARISON = False       # Same actor/seeds with vs without replay scout

TUNING_FILE = ROOT / "data/offline-corpus/macintosh-compute.json"
TUNED = json.loads(TUNING_FILE.read_text()) if TUNING_FILE.exists() else {}
WORKERS = TUNED.get("workers", min(8, max(1, (os.cpu_count() or 2) - 1)))  # override if desired
BACKEND = TUNED.get("backend", "cpu")  # set to "mps" to explicitly use Metal
COMPUTE = Compute(workers=WORKERS, backend=BACKEND, training_threads=4,
                  gpu_fraction=.5, min_available_gb=2, max_root_gb=100)
STAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
NEW_RESULTS = ROOT / "artifacts" / f"offline-ppo-{MODE}-{STAMP}"
# Prepared exploratory results: three seeds, five updates, original collection targets.
RESULTS = DATA_ROOT / "artifacts/offline-ppo-reddit-budget"
if not RESULTS.exists():
    RESULTS = DATA_ROOT / "artifacts/offline-ppo-pilot"
if not RESULTS.exists():
    RESULTS = DATA_ROOT / "artifacts/offline-ppo-smoke-v2"
SCOUT_ROOT = DATA_ROOT / "artifacts/offline-scout"
WORKER_BENCH_ROOT = DATA_ROOT / "artifacts/offline-worker-benchmark"
TRAIN_BENCH_ROOT = DATA_ROOT / "artifacts/offline-training-benchmark-final"

display(pd.DataFrame([asdict(a) for a in ARMS]))
display(asdict(COMPUTE))

In [ ]:
prepared = await prepare_teams(CORPUS, max_teams=4)
TEAMS = prepared["teams"]
display(pd.DataFrame([{"team": t["name"], "species": ", ".join(s["species"] for s in t["sets"])} for t in TEAMS]))
display(prepared["skipped"])

## Measure the hardware

Worker timings include spawn/startup and use identical seeded cases for every count. A short run can understate the benefit of a persistent pool. Training timings include CPU→device movement, collecting-likelihood validation, optimization, synchronization and saving. Each trial reloads the same frozen checkpoint and uses its compatible first-update episodes. Warm-up is excluded. A failed MPS likelihood check is reported rather than silently weakening that check.

Large timing trials repeat the measured first-update rollout shapes to reach at least 1,024 decisions. This repetition is restricted to throughput measurement; the learning matrix always consumes fresh trajectories once. The tables report both unique and timed decision counts.

In [ ]:
if RUN_WORKER_BENCHMARK:
    WORKER_BENCH_ROOT = ROOT / "artifacts" / f"offline-workers-{STAMP}"
    worker_results = benchmark_workers(WORKER_BENCH_ROOT, TEAMS, counts=(1, 4, 8, 12, 14), count_games=28)
elif (WORKER_BENCH_ROOT / "workers.json").exists():
    worker_results = json.loads((WORKER_BENCH_ROOT / "workers.json").read_text())
else:
    worker_results = []
if worker_results:
    worker_frame = pd.DataFrame(worker_results)
    display(worker_frame[["workers", "peak_wave_workers", "games", "seconds", "games_per_minute"]])
    worker_frame.plot.bar(x="workers", y="games_per_minute", legend=False, title="Offline simulator throughput")
    plt.ylabel("completed games / minute")
    plt.show()

## Run the PPO matrix

Learning rate and entropy ablations change one variable relative to the high-rate reference. The larger-collection and combined arms test the multi-setting recipe suggested in the thread. The repository control retains its default PPO coefficients. The entropy-decay arm is an additional experiment, not a confirmed recipe from the post.

The 480 and 1,024 targets correspond numerically to `8×60` and `8×128`, but collection finishes entire episodes and records overshoot. This collector does **not** reproduce truncated 60/128-step vector horizons. Random opposition trains every arm; separate random and tactical opponents evaluate it on fixed teams, alternating sides, and seeds distinct from collection. Periodic evaluation uses development cases; final evaluation uses another seed range. No candidate is automatically deployed.

In [ ]:
settings = {
    "smoke": dict(updates=1, target_steps=32, eval_games=4, seeds=(0,), eval_every=1),
    "pilot": dict(updates=3, target_steps=128, eval_games=8, seeds=(0,), eval_every=1),
    "full": dict(updates=30, target_steps=None, eval_games=100, seeds=(0, 1, 2), eval_every=5),
}[MODE]
if RUN_MATRIX:
    arms = tuple(replace(a, target_steps=settings["target_steps"]) if settings["target_steps"] else a for a in ARMS)
    print("Saving new matrix to", NEW_RESULTS)
    run_matrix(NEW_RESULTS, TEAMS, COMPUTE, arms=arms, seeds=settings["seeds"],
               updates=settings["updates"], eval_games=settings["eval_games"], eval_every=settings["eval_every"])
    RESULTS = NEW_RESULTS
print("Results directory:", RESULTS)
training, episodes, evaluation = tables(RESULTS)
display(training)
if not len(training):
    print("No saved updates yet. Set RUN_MATRIX=True to run the selected mode.")
if (RESULTS / "matrix.json").exists():
    plan = json.loads((RESULTS / "matrix.json").read_text())
    planned_runs = len(plan["arms"]) * len(plan["seeds"])
    done = len(list(RESULTS.glob("*/report.json")))
    print(f"Completed seed/arm runs: {done}/{planned_runs}; rerun this cell with RUN_MATRIX=False to refresh active results.")

In [ ]:
if len(episodes):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for (arm, seed), group in episodes.groupby(["arm", "seed"]):
        group = group.sort_values("training_steps")
        axes[0].plot(group.training_steps, group["return"].rolling(100, min_periods=1).mean(), label=f"{arm} / seed {seed}")
        axes[1].scatter(group.training_steps, group["return"], s=8, alpha=.35)
    axes[0].set(title="Completed episode return — rolling 100 games", xlabel="Accepted training decisions", ylabel="Mean terminal return", ylim=(-1.05, 1.05))
    axes[0].legend(fontsize=7)
    axes[1].set(title="Raw completed episode returns", xlabel="Accepted training decisions", ylabel="Terminal return")
    plt.tight_layout(); plt.show()
    display(episodes.groupby(["arm", "seed"]).agg(completed_games=("id", "count"), mean_return=("return", "mean"), return_std=("return", "std"), mean_decisions=("steps", "mean")))

In [ ]:
if len(training):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    for (arm, seed), group in training.groupby(["arm", "seed"]):
        for ax, metric in zip(axes.flat, ["loss", "mean_entropy", "approx_kl", "clip_fraction", "entropy_coef", "training_seconds"]):
            ax.plot(group.training_steps, group[metric], marker=".", label=f"{arm}/{seed}")
            ax.set(title=metric.replace("_", " "), xlabel="Accepted training decisions")
    axes[0, 2].axhline(.03, color="black", linestyle="--", alpha=.4, label="KL stop")
    axes[0, 0].legend(fontsize=6)
    plt.tight_layout(); plt.show()
    fig, ax = plt.subplots()
    for (arm, seed), group in training.groupby(["arm", "seed"]):
        ax.plot(group.elapsed_seconds / 60, group.training_steps, label=f"{arm}/{seed}")
    ax.set(title="End-to-end progress (includes evaluations)", xlabel="Elapsed minutes", ylabel="Accepted training decisions")
    ax.legend(fontsize=7); plt.show()

In [ ]:
if len(evaluation):
    display(evaluation[["arm", "seed", "label", "opponent", "completed", "unfinished", "mean_episode_return", "win_rate", "win_rate_wilson95"]])
    final = evaluation[evaluation.label == "final"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, opponent in zip(axes, ["random", "tactical"]):
        group = final[final.opponent == opponent]
        if len(group):
            lower = group.apply(lambda r: r.win_rate - r.win_rate_wilson95[0], axis=1)
            upper = group.apply(lambda r: r.win_rate_wilson95[1] - r.win_rate, axis=1)
            ax.errorbar(range(len(group)), group.win_rate, yerr=[lower, upper], fmt="o", capsize=4)
            ax.set_xticks(range(len(group)), [f"{r.arm}/{r.seed}" for r in group.itertuples()], rotation=65, ha="right", fontsize=7)
        ax.set(title=f"Final held-out win rate vs {opponent}", ylim=(0, 1), ylabel="Win rate; Wilson 95% interval")
    plt.tight_layout(); plt.show()
    print("Pilot intervals are wide. Judge learning against initial scores and independent seeds, not smoothness alone.")

In [ ]:
if RUN_TRAINING_BENCHMARK:
    first_arm = next(RESULTS.glob("*/initial.pt")).parent
    if DATA_ROOT != ROOT:
        from experiments.archive_results import restore_database
        restore_database(DATA_ROOT, (first_arm / "experience.sqlite3").relative_to(DATA_ROOT))
    TRAIN_BENCH_ROOT = ROOT / "artifacts" / f"offline-training-{STAMP}"
    training_benchmark = benchmark_training(TRAIN_BENCH_ROOT, first_arm, repeats=2)
elif (TRAIN_BENCH_ROOT / "training.json").exists():
    training_benchmark = json.loads((TRAIN_BENCH_ROOT / "training.json").read_text())
else:
    training_benchmark = []
if training_benchmark:
    display(pd.DataFrame(training_benchmark))
    valid = [r for r in training_benchmark if r["seconds"] is not None and not r["errors"]]
    if valid:
        best = min(valid, key=lambda r: r["seconds"])
        print("Fastest measured actual PPO trial:", best["backend"], "batch", best["minibatch_size"])
        print("For controlled learning comparisons, keep each arm's minibatch plan fixed when selecting backend.")

## Use the copied real battles

This independent scout experiment rebuilds labels from the raw public logs using the current conservative parser. Features contain only information revealed before the turn. Called moves, missing executed actions, and ambiguous switch-ins are excluded. Validation splits whole battles, using a stable digest bucket. Historical M-B can initialize shared scout weights; promotion still requires held-out M-C loss to improve. The source manifest retains licenses and exact regulation.

The optional matched-case comparison keeps actor weights, teams, seeds, sides and sampling unchanged and varies only the replay-trained scout. It tests whether the corpus helps this actor against the declared opponents. It does not turn public logs into human PPO trajectories.

In [ ]:
if RUN_SCOUT:
    if DATA_ROOT != ROOT:
        from experiments.archive_results import restore_database
        restore_database(DATA_ROOT, (CORPUS / "inputs/experience.sqlite3").relative_to(DATA_ROOT))
    SCOUT_ROOT = ROOT / "artifacts" / f"offline-scout-{STAMP}"
    scout_report = await prepare_scout(CORPUS, SCOUT_ROOT, epochs=3, backend=BACKEND)
elif (SCOUT_ROOT / "scout-report.json").exists():
    scout_report = json.loads((SCOUT_ROOT / "scout-report.json").read_text())
else:
    scout_report = []
if scout_report:
    display(pd.json_normalize(scout_report))
else:
    print("Set RUN_SCOUT=True to train from the copied real public battles.")

In [ ]:
COMPARISON_ROOT = DATA_ROOT / "artifacts/offline-scout-comparison"
if RUN_SCOUT_COMPARISON:
    checkpoint = CORPUS / "models" / (FMT + ".pt")
    COMPARISON_ROOT = ROOT / "artifacts" / f"offline-scout-comparison-{STAMP}"
    scout_comparison = compare_scout(COMPARISON_ROOT, TEAMS, COMPUTE, checkpoint, SCOUT_ROOT, count=100)
elif (COMPARISON_ROOT / "comparison.json").exists():
    scout_comparison = json.loads((COMPARISON_ROOT / "comparison.json").read_text())
else:
    scout_comparison = []
if scout_comparison:
    display(pd.DataFrame([{k: v for k, v in r.items() if k != "games"} for r in scout_comparison]))
if (COMPARISON_ROOT / "paired.json").exists():
    display(pd.DataFrame(json.loads((COMPARISON_ROOT / "paired.json").read_text())))

## Inspect historical ladder behavior

These exact recorded trajectories are diagnostic data from old collecting policies. The PPO matrix never reads this file as a training batch. Only the recorded pre-decision snapshots describe what was observable at that moment; final revealed sets must not be backfilled into earlier observations.

In [ ]:
historic = []
with gzip.open(CORPUS / "historic-ladder-episodes.jsonl.gz", "rt") as stream:
    for line in stream:
        e = json.loads(line)
        historic.append({"format": e["format"], "revision": e["revision"], "outcome": e["outcome"],
                         "decisions": len(e["steps"]), "team": e.get("team"), "created": e.get("created")})
history = pd.DataFrame(historic)
display(history.groupby("format").agg(battles=("outcome", "count"), mean_return=("outcome", "mean"), median_decisions=("decisions", "median")))
history.decisions.hist(bins=30)
plt.title("Recorded real ladder battle lengths"); plt.xlabel("Learner decisions"); plt.ylabel("Battles"); plt.show()
print("All reports, source fingerprints, checkpoints and per-game outcomes stay under gitignored local directories.")